# phase 3 — a layer-0 direction for coherent personas
See `PLAN.md`. New directions are fit at layer 0 against a persona objective (phase 1's layer-20 assistant direction, reversed) with an `H1` hinge, and against a 'not the stock opener' objective. Phase 1's own direction is reused at layer 20 as a reference arm. Phase 2's entropy direction is the other reference.

In [4]:
# === CELL 1 — rig (phase 2's) + inputs ===================================================================
import torch, torch.nn.functional as F, math, json, time, inspect, os
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
LN2 = math.log(2); Q = "what shall i do today"; MODEL_ID = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda:0"); model.eval(); model.requires_grad_(False)
dev, EOS, V = model.device, tokenizer.eos_token_id, model.get_input_embeddings().weight.shape[0]; CEIL = math.log2(V); D = model.config.hidden_size
_LTK = "logits_to_keep" if "logits_to_keep" in inspect.signature(model.forward).parameters else "num_logits_to_keep"
def _chat(t):
    try: return tokenizer.apply_chat_template([{"role":"user","content":t}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError: return tokenizer.apply_chat_template([{"role":"user","content":t}], tokenize=False, add_generation_prompt=True)
def scaffold(query):
    s = _chat(query); enc = tokenizer(s, add_special_tokens=False, return_offsets_mapping=True)
    ids, offs = enc["input_ids"], enc["offset_mapping"]; i0 = s.index(query); i1 = i0 + len(query)
    j0 = next(t for t,(a,b) in enumerate(offs) if b > i0); j1 = next(t for t,(a,b) in enumerate(offs) if a >= i1)
    return ids, (j0, j1)
def entropy_bits(lg):
    lp = F.log_softmax(lg.float(), -1); return -(lp.exp()*lp).sum(-1)/LN2
PROMPTS = json.loads('{"train": ["How do I get better at public speaking?", "Write a short poem about autumn leaves.", "What\'s the difference between a virus and a bacterium?", "Can you help me plan a two-day trip to Lisbon?", "Write a Python function that checks whether a string is a palindrome.", "How do I politely decline a meeting invitation?", "What causes the seasons on Earth?", "Give me five ideas for a birthday present for my dad.", "How does compound interest work?", "I can\'t focus when I work from home. Any advice?", "Explain what a hash map is to a beginner.", "Draft a friendly reminder email about an overdue invoice.", "What should I know before adopting a cat?", "Summarise the plot of Romeo and Juliet in three sentences.", "How do I fix a leaking tap?", "What\'s a good stretching routine for someone who sits all day?", "Why is the sky blue?", "Write a haiku about coffee.", "How do I convert Celsius to Fahrenheit?", "What\'s the best way to learn a new language as an adult?", "Recommend three podcasts about history.", "How do I make my sourdough starter more active?", "Explain the rules of chess briefly.", "What are some good questions to ask at the end of a job interview?"], "held_out": ["what shall i do today", "What shall I do today?", "What\'s a quick and healthy dinner I can make tonight?", "Give me a 3-sentence pep talk for a Monday morning.", "Explain how a rainbow forms, simply.", "Suggest a good book to read this weekend and why.", "What are three tips to sleep better?", "How do I write a good cover letter?", "What is machine learning, in one paragraph?", "Give me a simple recipe for pancakes.", "How can I save more money each month?", "Write a SQL query that returns the ten most recent orders.", "What\'s the capital of Australia, and why isn\'t it Sydney?", "Help me name my new golden retriever puppy.", "How long should I boil an egg for a runny yolk?", "Tell me a fun fact about octopuses."]}'); TRAIN, HELD = PROMPTS["train"], PROMPTS["held_out"]; ALL = TRAIN + HELD
SC = [scaffold(p) for p in ALL]; L = max(len(i) for i,_ in SC); N = len(ALL)
IDS = torch.full((N, L), EOS, dtype=torch.long); ATT = torch.zeros((N, L), dtype=torch.long); M_ALL = torch.zeros((N, L), dtype=torch.bool)
for r,(ids,(j0,j1)) in enumerate(SC):
    o = L - len(ids); IDS[r, o:] = torch.tensor(ids); ATT[r, o:] = 1; M_ALL[r, o:] = True
IDS, ATT, M_ALL = IDS.to(dev), ATT.to(dev), M_ALL.to(dev); POS = (ATT.cumsum(-1) - 1).clamp(min=0)
TR = torch.arange(N, device=dev) < len(TRAIN); HO = ~TR; iq = ALL.index(Q)
INP = np.load("/content/phase3_inputs.npz")
P1 = {k[4:]: torch.tensor(INP[k], device=dev) for k in INP.files if k.startswith("p1__")}
P2 = {k[4:]: torch.tensor(INP[k], device=dev) for k in INP.files if k.startswith("p2__")}
GAP = {"L20__massmean_early": 27.62, "L20__inlp_debiased_early": 6.886}   # assistant minus persona, from phase 1 meta
print(f"transformers {__import__('transformers').__version__} | prompts {len(TRAIN)}/{len(HELD)} | seq {L} | p1 dirs {len(P1)} | p2 dirs {list(P2)}")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

transformers 5.16.1 | prompts 24/16 | seq 27 | p1 dirs 22 | p2 dirs ['const_all_eps0.8', 'const_all_eps0.4']


In [5]:
# === CELL 2 — hooks: per-layer additive interventions, layer-20 capture ====================================
# INT[layer] = dict(u=unit vec or None, eps, mask [B,T] bool or None(all), always_on)
INT = {}; CAP = {"h20": None}
def _unit(x): return x / x.norm(dim=-1, keepdim=True).clamp(min=1e-6)
def _mk_hook(layer):
    def f(mod, inp, out):
        hs = out[0] if isinstance(out, tuple) else out
        B, T, _ = hs.shape
        if layer == 20 and T > 1: CAP["h20"] = hs[:, -1].float()          # last prompt position, before any L20 add
        st = INT.get(layer)
        if st is None or st["eps"] == 0.0: return out
        if T == 1 and not st["always_on"]: return out
        m = torch.ones((B, T), device=hs.device) if (T == 1 or st["mask"] is None) else st["mask"][:B, :T].float()
        hf = hs.float(); nrm = hf.norm(dim=-1, keepdim=True)
        new = (hf + st["eps"] * nrm * _unit(st["u"].float())[None, None, :] * m[..., None]).to(hs.dtype)
        return (new,) + tuple(out[1:]) if isinstance(out, tuple) else new
    return f
HOOKS = [model.model.layers[l].register_forward_hook(_mk_hook(l)) for l in [0, 16, 20]]
def set_int(layer, u, eps, mask="all", always_on=False):
    INT[layer] = dict(u=u, eps=eps, mask=(M_ALL if mask == "all" else mask), always_on=always_on)
def off(): INT.clear()
def fwd(rows=None):
    sl = slice(None) if rows is None else rows
    lg = model(input_ids=IDS[sl], attention_mask=ATT[sl], position_ids=POS[sl], **{_LTK:1}).logits[:, -1]
    return entropy_bits(lg), lg, CAP["h20"]
with torch.no_grad():
    off(); H_clean, LG_clean, H20_clean = fwd()
    PROJ_clean = {k: (H20_clean @ P1[k]) for k in GAP}
    TOP5 = LG_clean.float().softmax(-1).topk(5, -1)                  # each prompt's clean stock openers
print(f"clean H1 {H_clean[iq]:.4f} (phase 11 0.2366) | train {H_clean[TR].mean():.3f} held {H_clean[HO].mean():.3f}")
for k in GAP: print(f"  clean projection on {k}: mean {PROJ_clean[k].mean():.1f} (gap {GAP[k]}) | L20 resid norm {H20_clean.norm(dim=-1).mean():.1f}")
print("  stock openers:", [tokenizer.decode([int(t)]) for t in TOP5.indices[iq]], [round(float(p),3) for p in TOP5.values[iq]])


clean H1 0.2331 (phase 11 0.2366) | train 0.621 held 0.167
  clean projection on L20__massmean_early: mean 43.3 (gap 27.62) | L20 resid norm 118.9
  clean projection on L20__inlp_debiased_early: mean 1.3 (gap 6.886) | L20 resid norm 118.9
  stock openers: ['That', 'It', 'What', 'Ah', 'Great'] [0.974, 0.01, 0.007, 0.005, 0.001]


In [6]:
# === CELL 3 — objectives, fitting at layer 0, arms ==========================================================
HCAP, LAM_H, STEPS, LR = 8.0, 2.0, 150, 0.05
def stats(H): return dict(train_mean=float(H[TR].mean()), train_sd=float(H[TR].std()), held_mean=float(H[HO].mean()), held_sd=float(H[HO].std()), held_min=float(H[HO].min()), per_prompt=H.tolist())
RES = {"meta": dict(model=MODEL_ID, prompts=PROMPTS, hcap=HCAP, lam_h=LAM_H, steps=STEPS), "clean": stats(H_clean), "arms": {}}
def probe_obj(H, lg, h20, key):
    """minimise projection on the assistant direction, in gap units, with an H1 hinge."""
    proj = (h20 @ P1[key]) / GAP[key]
    return proj.mean() + LAM_H * F.relu(H - HCAP).mean(), proj
def opener_obj(H, lg, h20, key=None):
    """minimise the total probability of each prompt's clean top-5 first tokens, with an H1 hinge."""
    p = lg.float().softmax(-1).gather(-1, TOP5.indices[TR]).sum(-1)
    return torch.log(p + 1e-6).mean() + LAM_H * F.relu(H - HCAP).mean(), p
def fit(obj, key, eps, seed=0):
    g = torch.Generator(device="cpu").manual_seed(seed)
    u = torch.randn(D, generator=g).to(dev).requires_grad_(True)
    opt = torch.optim.Adam([u], lr=LR); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, STEPS)
    best = (1e9, None); hist = []
    for it in range(STEPS):
        set_int(0, u, eps)
        with torch.enable_grad():
            H, lg, h20 = fwd(TR); loss, aux = obj(H, lg, h20, key)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        hist.append((float(loss), float(H.mean()), float(aux.mean())))
        if float(loss) < best[0]: best = (float(loss), u.detach().clone())
    off(); return best[1], hist
def evaluate(name, **kw):
    with torch.no_grad():
        H, lg, h20 = fwd(); off()
        proj = {k: float(((h20 @ P1[k]) / GAP[k]).mean()) for k in GAP}
        pop = float(lg.float().softmax(-1).gather(-1, TOP5.indices).sum(-1).mean())
    RES["arms"][name] = dict(stats(H), proj_gap_units=proj, p_openers=pop, **kw)
    s = RES["arms"][name]
    print(f"{name:36s} H1 train {s['train_mean']:5.2f} held {s['held_mean']:5.2f} (min {s['held_min']:5.2f}) | proj/gap mm {proj['L20__massmean_early']:+.2f} inlp {proj['L20__inlp_debiased_early']:+.2f} | p(openers) {pop:.3f}")
FIT = {}
off(); evaluate("clean")
set_int(0, P2["const_all_eps0.8"], 0.8); evaluate("p2-entropy·L0·eps0.8", layer=0, eps=0.8)
set_int(0, P2["const_all_eps0.4"], 0.4); evaluate("p2-entropy·L0·eps0.4", layer=0, eps=0.4)
g = torch.Generator(device="cpu").manual_seed(0); RANDU = [torch.randn(D, generator=g).to(dev) for _ in range(8)]
for layer in [0, 20]:
    Hs = []
    for u in RANDU:
        set_int(layer, u, 0.4);
        with torch.no_grad(): Hs.append(fwd()[0]); off()
    Hs = torch.stack(Hs); RES["arms"][f"rand·L{layer}·eps0.4"] = dict(stats(Hs.mean(0)), layer=layer, eps=0.4, n_draws=8)
    print(f"rand·L{layer}·eps0.4                      H1 train {Hs[:, TR].mean():5.2f} held {Hs[:, HO].mean():5.2f} (max draw {Hs.max():.2f})")
# arm 1: phase 1's assistant direction, reversed (negative eps), at L20, prompt-only here (always-on only matters for rollouts)
for key in ["L20__massmean_early", "L20__inlp_debiased_early"]:
    for e in [0.2, 0.35, 0.5]:
        set_int(20, P1[key], -e); evaluate(f"p1-assist⁻·{key.split('__')[1]}·L20·eps{e}", layer=20, eps=-e, key=key)
# arm 2 + 3: fit at layer 0
t0 = time.time()
for key in ["L20__massmean_early", "L20__inlp_debiased_early"]:
    for e in [0.2, 0.4]:
        name = f"probe·{key.split('__')[1]}·L0·eps{e}"; u, hist = fit(probe_obj, key, e); FIT[name] = u
        set_int(0, u, e); evaluate(name, layer=0, eps=e, key=key, curve=hist[::10])
        print(f"      {time.time()-t0:5.0f}s  (loss, H1, proj) every 30: " + " ".join(f"({a:.2f},{b:.1f},{c:+.2f})" for a,b,c in hist[::30]))
for e in [0.2, 0.4]:
    name = f"opener·L0·eps{e}"; u, hist = fit(opener_obj, None, e); FIT[name] = u
    set_int(0, u, e); evaluate(name, layer=0, eps=e, curve=hist[::10])
    print(f"      {time.time()-t0:5.0f}s  (loss, H1, p_open) every 30: " + " ".join(f"({a:.2f},{b:.1f},{c:.2f})" for a,b,c in hist[::30]))
np.savez("/content/phase3_directions.npz", **{n: u.cpu().float().numpy() for n, u in FIT.items()})
json.dump(RES, open("/content/phase3_results.json", "w")); print("saved")


clean                                H1 train  0.62 held  0.17 (min  0.00) | proj/gap mm +1.57 inlp +0.18 | p(openers) 0.999
p2-entropy·L0·eps0.8                 H1 train 16.34 held 16.23 (min 15.82) | proj/gap mm +1.15 inlp +0.22 | p(openers) 0.000
p2-entropy·L0·eps0.4                 H1 train 14.65 held 13.37 (min  7.87) | proj/gap mm +0.58 inlp +0.14 | p(openers) 0.002
rand·L0·eps0.4                      H1 train  0.65 held  0.21 (max draw 2.19)
rand·L20·eps0.4                      H1 train  0.78 held  0.73 (max draw 3.33)
p1-assist⁻·massmean_early·L20·eps0.2 H1 train  0.43 held  0.49 (min  0.00) | proj/gap mm +1.57 inlp +0.18 | p(openers) 0.926
p1-assist⁻·massmean_early·L20·eps0.35 H1 train  0.66 held  0.74 (min  0.00) | proj/gap mm +1.57 inlp +0.18 | p(openers) 0.789
p1-assist⁻·massmean_early·L20·eps0.5 H1 train  1.17 held  1.47 (min  0.00) | proj/gap mm +1.57 inlp +0.18 | p(openers) 0.391
p1-assist⁻·inlp_debiased_early·L20·eps0.2 H1 train  0.84 held  0.73 (min  0.00) | proj/gap m

/tmp/ipykernel_1219/4131016648.py:23: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  hist.append((float(loss), float(H.mean()), float(aux.mean())))


probe·massmean_early·L0·eps0.2       H1 train  1.65 held  0.83 (min  0.00) | proj/gap mm -2.21 inlp +0.13 | p(openers) 0.567
         29s  (loss, H1, proj) every 30: (1.58,0.6,+1.58) (-0.09,0.8,-0.09) (-1.19,1.1,-1.19) (-1.88,1.0,-1.88) (-2.35,1.4,-2.35)
probe·massmean_early·L0·eps0.4       H1 train  3.66 held  2.57 (min  0.00) | proj/gap mm -4.16 inlp -0.38 | p(openers) 0.118
         58s  (loss, H1, proj) every 30: (1.58,0.6,+1.58) (-1.35,1.0,-1.35) (-2.80,1.1,-2.80) (-3.83,2.8,-3.83) (-4.30,3.7,-4.30)
probe·inlp_debiased_early·L0·eps0.2  H1 train  2.04 held  1.41 (min  0.00) | proj/gap mm +0.20 inlp -4.75 | p(openers) 0.142
         87s  (loss, H1, proj) every 30: (0.20,0.6,+0.20) (-1.06,0.7,-1.06) (-3.22,1.2,-3.22) (-4.72,1.2,-4.72) (-5.18,1.8,-5.18)
probe·inlp_debiased_early·L0·eps0.4  H1 train  5.44 held  6.15 (min  2.26) | proj/gap mm -1.03 inlp -17.32 | p(openers) 0.001
        116s  (loss, H1, proj) every 30: (0.23,0.6,+0.23) (-3.41,1.2,-3.41) (-11.17,5.0,-11.17) (-15.74,5.4,-

In [7]:
# === CELL 4 — rollouts: 10 seeds x 8 held-out prompts x 96 tokens per arm ==================================
ROLL_PROMPTS = HELD[:8]; SEEDS = 10; NEW = 96
def script_of(text):
    cnt = {}
    for ch in text:
        if not ch.isalpha(): continue
        o = ord(ch)
        k = ("latin" if o < 0x250 else "cyrillic" if 0x400 <= o < 0x530 else "arabic" if 0x600 <= o < 0x700 else
             "hangul" if 0xAC00 <= o < 0xD7B0 or 0x1100 <= o < 0x1200 else "kana" if 0x3040 <= o < 0x3100 else
             "han" if 0x4E00 <= o < 0xA000 or 0x3400 <= o < 0x4DC0 else "other")
        cnt[k] = cnt.get(k, 0) + 1
    return max(cnt, key=cnt.get) if cnt else "none"
def rollout(specs, prompt, seed):
    """specs: list of (layer, u, eps, always_on). Intervention on all prompt positions; always_on also hits decode steps."""
    ids, _ = scaffold(prompt); T = len(ids)
    inp = torch.tensor([ids] * SEEDS, device=dev); att = torch.ones_like(inp); m = torch.ones((SEEDS, T), dtype=torch.bool, device=dev)
    off()
    for (layer, u, eps, always_on) in specs: set_int(layer, u, eps, mask=m, always_on=always_on)
    torch.manual_seed(seed)
    with torch.no_grad():
        out = model.generate(input_ids=inp, attention_mask=att, do_sample=True, temperature=1.0, top_p=1.0, top_k=0,
                             max_new_tokens=NEW, output_logits=True, return_dict_in_generate=True, pad_token_id=EOS)
    off()
    Hs = torch.stack([entropy_bits(l) for l in out.logits], 1); gen = out.sequences[:, T:]; res = []
    for s in range(SEEDS):
        g = gen[s].tolist(); n = g.index(EOS) if EOS in g else len(g); g = g[:n]
        txt = tokenizer.decode(g, skip_special_tokens=True); ng = [tuple(g[i:i+4]) for i in range(max(0, len(g)-3))]
        res.append(dict(seed=s, H_first=float(Hs[s, 0]), Hbar=float(Hs[s, 1:max(n,2)].mean()), n_tok=n, uniq4=(len(set(ng)) / max(1, len(ng))), script=script_of(txt), text=txt))
    return res
ARMS = {
  "clean": [],
  "p2-entropy·L0·eps0.8": [(0, P2["const_all_eps0.8"], 0.8, False)],
  "rand·L0·eps0.4": [(0, RANDU[0], 0.4, False)],
  "rand·L20·eps0.4·on": [(20, RANDU[1], 0.4, True)],
  "p1-assist⁻·mm·L20·eps0.35·prompt": [(20, P1["L20__massmean_early"], -0.35, False)],
  "p1-assist⁻·mm·L20·eps0.35·on": [(20, P1["L20__massmean_early"], -0.35, True)],
  "p1-assist⁻·mm·L20·eps0.5·on": [(20, P1["L20__massmean_early"], -0.5, True)],
  "p1-assist⁻·inlp·L20·eps0.35·on": [(20, P1["L20__inlp_debiased_early"], -0.35, True)],
  "probe·mm·L0·eps0.2": [(0, FIT["probe·massmean_early·L0·eps0.2"], 0.2, False)],
  "probe·mm·L0·eps0.4": [(0, FIT["probe·massmean_early·L0·eps0.4"], 0.4, False)],
  "probe·inlp·L0·eps0.4": [(0, FIT["probe·inlp_debiased_early·L0·eps0.4"], 0.4, False)],
  "opener·L0·eps0.2": [(0, FIT["opener·L0·eps0.2"], 0.2, False)],
  "opener·L0·eps0.4": [(0, FIT["opener·L0·eps0.4"], 0.4, False)],
}
RES["rollouts"] = {}
for n, specs in ARMS.items():
    rows = []
    for k, pr in enumerate(ROLL_PROMPTS): rows += [dict(prompt=pr, **r) for r in rollout(specs, pr, seed=5000+k)]
    RES["rollouts"][n] = rows; sc = {}
    for r in rows: sc[r["script"]] = sc.get(r["script"], 0) + 1
    print(f"{n:36s} H_first {np.mean([r['H_first'] for r in rows]):5.2f} | Hbar {np.mean([r['Hbar'] for r in rows]):4.2f} | uniq4 {np.mean([r['uniq4'] for r in rows]):.2f} | len {np.mean([r['n_tok'] for r in rows]):4.1f} | {sc}")
    for r in rows[:1] + rows[10:11] + rows[30:31]: print(f"      [{r['prompt'][:26]!r} H1 {r['H_first']:.1f} Hbar {r['Hbar']:.1f}] {r['text'][:150]!r}")
json.dump(RES, open("/content/phase3_results.json", "w"), ensure_ascii=False); print("saved")


clean                                H_first  0.27 | Hbar 0.49 | uniq4 0.99 | len 91.0 | {'latin': 80}
      ['what shall i do today' H1 0.2 Hbar 0.7] "That's a great question! What you do today depends on your mood, goals, and what you're passionate about. Here are a few ideas to help you decide:\n\n##"
      ['What shall I do today?' H1 0.2 Hbar 0.6] "That's a great question! What you do today depends on your goals, mood, and what you enjoy. Here are a few ideas to inspire you:\n\n### 1. **Do Somethin"
      ['Give me a 3-sentence pep t' H1 0.0 Hbar 0.3] "Monday mornings can be tough, but they're also full of potential. Take a deep breath, set your intentions for the week, and remember that every challe"
p2-entropy·L0·eps0.8                 H_first 16.26 | Hbar 2.19 | uniq4 0.97 | len 92.0 | {'han': 44, 'latin': 23, 'other': 6, 'kana': 6, 'cyrillic': 1}
      ['what shall i do today' H1 16.2 Hbar 1.3] ' multiline\n\n</think>\n\n看起来你可能是在问“今天该做什么”或者类似的句子，但句子中的“什麼”和“我”之间似乎有些混乱。在中文里，“今天我